# Sunshine-Gated Closed-Case Pipeline

Closed-case true-crime research pipeline for documentary production.

## Setup Checklist

Before running, add these **Colab Secrets** (key icon in left sidebar):

| Secret Name | Where to get it |
|---|---|
| `YOUTUBE_API_KEY` | [Google Cloud Console](https://console.cloud.google.com/apis/credentials) — enable YouTube Data API v3 |
| `BRAVE_API_KEY` | [Brave Search API](https://brave.com/search/api/) |
| `OPENROUTER_API_KEY` | [OpenRouter](https://openrouter.ai/keys) |
| `GOOGLE_SHEETS_ID` | The spreadsheet ID from your Google Sheet URL (`docs.google.com/spreadsheets/d/{THIS_PART}/edit`) |

Then run cells **top to bottom**. Each stage can be run independently.

## Cell 1 — Install & Import

In [ ]:
import os, subprocess, sys

REPO_DIR = "/content/FlameOn"
BRANCH = "claude/crime-research-pipeline-p3oci"

# Clone if needed, then checkout the correct branch
if not os.path.exists(os.path.join(REPO_DIR, "src")):
    subprocess.run(["rm", "-rf", REPO_DIR], check=False)
    subprocess.run(
        ["git", "clone", "-b", BRANCH, "https://github.com/jj55222/FlameOn.git", REPO_DIR],
        check=True,
    )
    print(f"Cloned branch {BRANCH}")
else:
    # Already cloned — make sure we're on the right branch and up to date
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)
    print(f"Updated to latest {BRANCH}")

# Install dependencies using absolute path
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", os.path.join(REPO_DIR, "requirements.txt")], check=True)

# Add to Python path
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Verify imports
from src.main import (
    run_pipeline, load_channels,
    stage_intake, stage_validate, stage_discover, stage_download,
)
from src.sheet import SheetRegistry, SCOPES
from src.storage import PipelineStorage
from src.models import ChannelConfig, CaseCandidate, SHEET_COLUMNS
from src.logger import setup_logger, get_logger

print("All imports successful.")

## Cell 2 — Load API Keys from Colab Secrets

In [ ]:
from google.colab import userdata

PIPELINE_ROOT = "/content/CrimeDoc-Pipeline"

config = {
    # API keys from Colab Secrets (strip whitespace to avoid header errors)
    "youtube_api_key": (userdata.get("YOUTUBE_API_KEY") or "").strip(),
    "brave_api_key": (userdata.get("BRAVE_API_KEY") or "").strip(),
    "openrouter_api_key": (userdata.get("OPENROUTER_API_KEY") or "").strip(),

    # Google Sheets (auth handled separately via Colab auth)
    "google_sheets_spreadsheet_id": (userdata.get("GOOGLE_SHEETS_ID") or "").strip(),
    "google_sheets_tab_name": "CaseRegistry",

    # Local storage
    "pipeline_root": PIPELINE_ROOT,

    # OpenRouter model config (change these if you want a different model)
    "openrouter_model_extraction": "google/gemini-flash-1.5",
    "openrouter_model_validation": "google/gemini-flash-1.5",
    "openrouter_base_url": "https://openrouter.ai/api/v1",

    # Rate limiting
    "youtube_requests_per_second": 1,
    "brave_requests_per_second": 1,
    "download_requests_per_second": 0.5,

    # Intake
    "max_videos_per_channel": 50,

    # Validation
    "max_brave_queries_per_case": 3,

    # Downloads
    "max_download_size_mb": 100,
    "download_timeout_seconds": 60,
}

# Verify keys are present (not empty)
for key in ["youtube_api_key", "brave_api_key", "openrouter_api_key", "google_sheets_spreadsheet_id"]:
    val = config[key]
    status = "OK" if val else "MISSING"
    print(f"  {key}: {status}")

print(f"\nPipeline root: {PIPELINE_ROOT}")

## Cell 3 — Google Sheets Auth (Colab native)

This uses Colab's built-in Google auth — no service account JSON file needed.  
A popup will ask you to authorize access to your Google account.

In [ ]:
import sys
sys.path.insert(0, '/content/FlameOn')

from google.colab import auth
import google.auth
from src.sheet import SheetRegistry

auth.authenticate_user()

SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive",
]
creds, _ = google.auth.default(scopes=SCOPES)

sheet = SheetRegistry.from_credentials(
    credentials=creds,
    spreadsheet_id=config["google_sheets_spreadsheet_id"],
    tab_name=config["google_sheets_tab_name"],
)
sheet.ensure_headers()
print("Google Sheets connected and headers verified.")

## (Optional) Wipe Sheet Before Re-run

Run this cell to **clear all data rows** from the sheet (keeps headers).  
Use this when you want a fresh run after code changes.

In [ ]:
# Wipe all data rows from the sheet (keeps headers intact)
sheet.clear_all_data()
print("Sheet wiped. Ready for a fresh run.")

## Cell 4 — Initialize Storage & Logging

In [ ]:
import os, sys
sys.path.insert(0, '/content/FlameOn')

from src.storage import PipelineStorage
from src.logger import setup_logger, get_logger
from src.main import load_channels

setup_logger(PIPELINE_ROOT)
log = get_logger()

storage = PipelineStorage(PIPELINE_ROOT)
storage.init_dirs()

channels = load_channels("/content/FlameOn/config/channels.yaml")
print(f"Loaded {len(channels)} agency channels")
print(f"Pipeline directories created at {PIPELINE_ROOT}")

for d in sorted(os.listdir(PIPELINE_ROOT)):
    print(f"  {d}/")

---
## Stage 1 — YouTube Intake

Fetches recent uploads from all tracked agency channels.  
Extracts suspect names, dates, and case keywords.  
Deduplicates by video_id against existing Sheet rows.

In [ ]:
import sys
sys.path.insert(0, '/content/FlameOn')
from src.main import stage_intake

candidates = stage_intake(config, channels, sheet, storage)

print(f"\n{'='*60}")
print(f"Intake results: {len(candidates)} new candidates")
print(f"{'='*60}")

if candidates:
    import pandas as pd
    df = pd.DataFrame([{
        "case_id": c.case_id,
        "suspect_name": c.suspect_name or "(none)",
        "agency": c.agency_name,
        "state": c.state,
        "keywords": c.case_keywords[:50] if c.case_keywords else "(none)",
        "title": c.video_title[:60],
    } for c in candidates])
    display(df)
else:
    print("No new candidates found (all may be duplicates of existing entries).")

---
## Stage 2 — Cheap Validation Gate

Searches Brave for each candidate. Uses OpenRouter LLM to determine if the case is **closed/sentenced**.  
Routes to: `validated_closed`, `rejected_open_or_unconfirmed`, or `manual_review`.

In [ ]:
import sys
sys.path.insert(0, '/content/FlameOn')
from src.main import stage_validate

# Use candidates from Stage 1 if available, otherwise load from sheet
validated = stage_validate(config, sheet, storage, candidates if 'candidates' in dir() else None)

print(f"\n{'='*60}")
print(f"Validation results: {len(validated)} cases validated as CLOSED")
print(f"{'='*60}")

if validated:
    import pandas as pd
    df = pd.DataFrame([{
        "case_id": c.case_id,
        "suspect_name": c.suspect_name,
        "state": c.state,
        "validation_note": c.validation_note[:80],
        "source_rank": c.source_rank_used,
        "folder": c.local_case_folder,
    } for c in validated])
    display(df)
else:
    print("No cases passed validation. Check the Sheet for rejection reasons.")

---
## Stage 3A — Link Discovery

For validated cases only: searches for court dockets, sentencing orders,  
news articles, and BWC/interrogation footage using state-aware logic.

In [ ]:
import sys
sys.path.insert(0, '/content/FlameOn')
from src.main import stage_discover

# Use validated from Stage 2 if available, otherwise load from sheet
discovered = stage_discover(config, sheet, storage, validated if 'validated' in dir() else None)

print(f"\n{'='*60}")
print(f"Discovery complete for {len(discovered)} cases")
print(f"{'='*60}")

if discovered:
    import pandas as pd
    df = pd.DataFrame([{
        "case_id": c.case_id,
        "suspect_name": c.suspect_name,
        "links_found": c.links_discovered,
        "official_corroboration": c.official_corroboration_status,
    } for c in discovered])
    display(df)

---
## Stage 3B — Asset Download

Downloads recommended assets (court PDFs, BWC video links) from the link inventory.  
Only downloads links marked `download_recommended: true`.

In [ ]:
import os, sys
sys.path.insert(0, '/content/FlameOn')
from src.main import stage_download

# Use discovered from Stage 3A if available, otherwise load from sheet
stage_download(config, sheet, storage, discovered if 'discovered' in dir() else None)

print(f"\n{'='*60}")
print("Downloads complete. Check the Sheet for results.")
print(f"{'='*60}")

dl_root = os.path.join(PIPELINE_ROOT, "02_Validated-Closed")
if os.path.exists(dl_root):
    for case_dir in sorted(os.listdir(dl_root)):
        dl_dir = os.path.join(dl_root, case_dir, "downloads")
        if os.path.exists(dl_dir):
            files = os.listdir(dl_dir)
            if files:
                print(f"\n{case_dir}/downloads/")
                for f in files:
                    size = os.path.getsize(os.path.join(dl_dir, f))
                    print(f"  {f} ({size:,} bytes)")

---
## (Optional) Run Full Pipeline in One Call

If you want to run all stages at once instead of cell-by-cell:

In [ ]:
# Uncomment to run the full pipeline:
#
# results = run_pipeline(
#     config=config,
#     stages=["intake", "validate", "discover", "download"],
#     sheet=sheet,
#     channels=channels,
#     pipeline_root=PIPELINE_ROOT,
# )
# print(f"Candidates: {len(results['candidates'] or [])}")
# print(f"Validated:  {len(results['validated'] or [])}")
# print(f"Discovered: {len(results['discovered'] or [])}")

---
## (Optional) Mount Google Drive for Persistence

Colab's filesystem is ephemeral. Mount Drive to keep results across sessions.

In [ ]:
# Uncomment to mount Drive and copy results:
#
# from google.colab import drive
# drive.mount('/content/drive')
#
# import shutil
# dest = '/content/drive/MyDrive/CrimeDoc-Pipeline'
# shutil.copytree(PIPELINE_ROOT, dest, dirs_exist_ok=True)
# print(f"Pipeline data copied to {dest}")

---
## (Optional) Inspect a Specific Case

In [ ]:
# View a specific case's validation log and link inventory
#
# import json
# case_folder = "FL_Smith_Sentenced_2024"  # <-- change this
# base = os.path.join(PIPELINE_ROOT, "02_Validated-Closed", case_folder)
#
# # Validation log
# vlog = os.path.join(base, "validation_log.txt")
# if os.path.exists(vlog):
#     print("=== VALIDATION LOG ===")
#     print(open(vlog).read())
#
# # Link inventory
# inv = os.path.join(base, "links_inventory.json")
# if os.path.exists(inv):
#     print("\n=== LINK INVENTORY ===")
#     data = json.load(open(inv))
#     for link in data.get('links', []):
#         print(f"  [{link['source_class']}] {link['link_type']}: {link['url'][:80]}")